<a href="https://colab.research.google.com/github/grmntfrancis0/earthengine-community/blob/main/Population_estimate_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade xee

In [ ]:
!pip install -U geemap

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project = "ee-grmntfrancis0",
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
import geemap

In [ ]:
map = geemap.Map()
map

In [ ]:
roi = ee.Geometry.Point(map.draw_last_feature.geometry().coordinates().getInfo())
roi

In [ ]:
border = (ee.FeatureCollection("FAO/GAUL/2015/level1")

  .filterBounds(roi)
  .geometry()
  .simplify(100)
)
vis_params = {"color":"yellow"}
map.addLayer(border, vis_params, "border")

In [ ]:
pop = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
pop

In [ ]:
pop_stack = pop.toBands().clip(roi)
pop_stack

In [ ]:
import geemap

map = geemap.Map()
map.add("basemap_selector")
map.addLayer(pop_stack,{},'pop_stack')

In [ ]:
def pop_count(img):
  pop_sum = img.reduceRegion(reducer = ee.Reducer.sum(), geometry = border, scale = 100, bestEffort=True, maxPixels=1e9).values().get(0)
  date = img.date().format('YYYY-MM-dd')
  return ee.Feature(None, {'date': date, 'pop': pop_sum})

In [ ]:
pop_val = pop.map(pop_count)
feature_list = pop_val.aggregate_array('pop').getInfo()
date_list = pop_val.aggregate_array('date').getInfo()
pop_val

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
pop_sum = feature_list
pop_sum

In [ ]:
import matplotlib.pyplot as plt

# Create a DataFrame
df = pd.DataFrame({'Date': date_list, 'Population': pop_sum})

# Convert 'Date' column to datetime objects
df['Date'] = pd.to_datetime(df['Date'])

# Plot the population over time
plt.figure(figsize=(10, 6))
plt.plot(df['Date'], df['Population'], marker='o')
plt.title('Eastern Equatoria State Population Over Time')
plt.xlabel('Date')
plt.ylabel('Population in Millions')
plt.grid(True, alpha = 0.3, color = 'grey',)
plt.show()

In [ ]:
# Set 'Date' as the index for calculations
df_index = df.set_index('Date')

# Calculate the population growth rate (percentage change)
df_index['growth_rate'] = df_index['Population'].pct_change() * 100

# Smooth the growth rate using a rolling mean (e.g., a 2-period window)
growth_rate_smoothed = df_index['growth_rate'].rolling(window=2, min_periods=1).mean()

# Display the DataFrame with growth rates
display(df_index)

In [ ]:
# Plotting Growth Rate (Original vs. Smoothed)

plt.figure(figsize=(10, 6))
df_index['growth_rate'].plot(label='Population Change', legend=True)
growth_rate_smoothed.plot(label='Population Change Smoothed', legend=True)
plt.title('Eastern Eastern State Population Change 1980-2030')
plt.xlabel('Date')
plt.ylabel('Population Change 1980-2030')
plt.grid(True, alpha = 0.3)
plt.show()

# To change the font type, you can set the 'font.family' parameter
# Common options include 'serif', 'sans-serif', 'monospace', 'cursive', 'fantasy'
# For example, to set a serif font:
plt.rcParams.update({'font.family': 'serif'})

print(f"Current font family: {plt.rcParams['font.family']}")

plt.savefig("ees_pop.png", dpi = 365, bbox_inches = 'tight')

In [ ]:
import matplotlib.pyplot as plt

# Set the global font size
plt.rcParams.update({'font.size': 15}) # You can change this value to your preference

In [ ]:
# To change the font type, you can set the 'font.family' parameter
# Common options include 'serif', 'sans-serif', 'monospace', 'cursive', 'fantasy'
# For example, to set a serif font:
plt.rcParams.update({'font.family': 'serif'})

print(f"Current font family: {plt.rcParams['font.family']}")

In [ ]:
import matplotlib.pyplot as plt

# Create figure and primary axis
fig, ax1 = plt.subplots(figsize=(12, 7))

# Plot Population Growth Rate on the primary y-axis
ax1.plot(df_index.index, df_index['growth_rate'], marker='o', color='tab:red', label='Population Growth Rate (%)')
ax1.set_xlabel('Date')
ax1.set_ylabel('Population Growth Rate (%)', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.grid(True, alpha=0.3, color='grey', linestyle='--')

# Create a secondary y-axis for Population
ax2 = ax1.twinx()
# Plot Population (scaled to millions) on the secondary y-axis
ax2.plot(df['Date'], df['Population'] / 1_000_000, marker='x', color='tab:blue', label='Population (Millions)')
ax2.set_ylabel('Population (Millions)', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

# Title and legend
plt.title('Eastern Equatoria State Population & Growth Rate Over Time')
fig.legend(loc='upper left', bbox_to_anchor=(0.15, 0.9))

plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()


**Reasoning**:
The subtask requires reviewing the `growth_rate` and `growth_rate_smoothed` columns. Displaying the `df_index` DataFrame will allow for a direct review of these values.



In [ ]:
display(df_index)

### Analysis of Population Growth Rates

Based on the `df_index` DataFrame and the `growth_rate_smoothed` series, the following observations can be made regarding the population growth rate in Eastern Equatoria State:

1.  **Early Growth (1980-1995)**: The growth rate started at approximately 68.7% in 1980 and generally increased, reaching around 99.9% by 1995. The smoothed rate also shows a steady increase during this period.

2.  **Period of Rapid Acceleration (2000-2010)**: The population growth rate experienced its most significant acceleration, peaking around 2005-2010. The original growth rate reached its highest at 127.22% in 2005, while the smoothed rate showed its peak at 125.76% in 2010. This indicates an extremely rapid increase in population during this decade.

3.  **Recent Deceleration (2015-2030)**: Following the peak, the growth rate began to decelerate. From 2015 onwards, both the original and smoothed growth rates show a noticeable decline. By 2030, the projected growth rate is around 38.2% (original) and 43.56% (smoothed). While these rates are significantly lower than the peak, they still represent substantial percentage increases year over year.

4.  **Dynamics of Change**: The trend indicates a period of increasing acceleration in population growth from 1980 to around 2010, followed by a period of deceleration. Despite the deceleration, the actual population continues to grow substantially due to the large base population reached during the rapid acceleration phase.

In summary, Eastern Equatoria State has seen a strong acceleration in population growth rates that peaked around 2005-2010, followed by a projected deceleration, though growth remains high.

### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.


### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.


### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.


### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.

### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.


```markdown
### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.
```

### Comprehensive Summary of Population Trends and Growth Rates for Eastern Equatoria State (1975-2030)

This analysis combines observations from both the overall population trends and the calculated growth rates for Eastern Equatoria State from 1975 to 2030.

**Overall Population Trend:**
Eastern Equatoria State is projected to experience a dramatic and sustained increase in its population over the analyzed period. Starting from an estimated 11,392 in 1975, the population is forecasted to reach approximately 7.7 million by 2030. This represents an exponential growth pattern, particularly accelerating in the 21st century.

**Periods of Significant Increase:**
*   **Early Growth (1975-1990):** The population showed steady, though modest, growth, moving from tens of thousands to roughly 64,000.
*   **Accelerated Growth (1995-2005):** A significant acceleration began, with the population crossing 100,000 by 1995 and exceeding 600,000 by 2005.
*   **Rapid Expansion (2010-2030):** This period marks the most rapid increase, with the population entering the millions. From 2010 (approx. 1.37 million) to 2015 (approx. 2.61 million), the population nearly doubled. This rapid growth is projected to continue, reaching over 7.7 million by 2030.

**Dynamics of Population Growth Rate:**
*   **Initial Acceleration (1980-1995):** The annual growth rate, as represented by percentage change, steadily increased from about 68.7% in 1980 to nearly 100% by 1995.
*   **Peak Growth Rate (2000-2010):** The period between 2000 and 2010 witnessed the highest rates of population growth. The original growth rate peaked at an impressive 127.22% in 2005, with the smoothed rate showing a peak around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this decade.
*   **Projected Deceleration (2015-2030):** Following this peak, the growth rate began a deceleration phase. By 2030, the projected growth rate, while still substantial, is expected to be around 38-43%, significantly lower than its peak. This deceleration suggests that while the population continues to grow, the rate at which it expands is slowing down.

**Key Insights and Conclusions:**
Eastern Equatoria State has undergone a demographic transformation characterized by exceptionally high population growth. The most critical period of acceleration occurred between 2000 and 2010, resulting in a large base population. Although the growth rate is projected to decelerate in the coming decades, the sheer momentum from previous high growth means that the absolute population numbers will continue to increase substantially. This sustained rapid population expansion has significant implications for resource allocation, infrastructure development, and social services within the region.


### Analysis of Population Growth Rates

**1. Review of `growth_rate` column:**

The `growth_rate` column in the `df_index` DataFrame shows the percentage change in population between consecutive periods.

*   **Periods of High Growth:** The growth rate generally increased from 1975 to 2005, peaking around 2005-2010 with growth rates exceeding 120%. This indicates a rapid acceleration in population increase during this timeframe.
    *   `1975-1980`: 68.75%
    *   `1980-1985`: 85.43%
    *   `1985-1990`: 79.51%
    *   `1990-1995`: 99.98%
    *   `1995-2000`: 109.61%
    *   `2000-2005`: 127.22% (Peak)
    *   `2005-2010`: 124.30%

*   **Periods of Decelerating Growth:** After 2010, the growth rate began to decrease:
    *   `2010-2015`: 90.74%
    *   `2015-2020`: 43.54%
    *   `2020-2025`: 48.92%
    *   `2025-2030`: 38.20% (Lowest projected growth rate in recent periods)

*   **Fluctuations:** While the overall trend was upward until 2010 and then downward, there are minor fluctuations (e.g., slight dip from 1985-1990, and a small increase from 2015-2020, then a dip again).

**2. Review of `growth_rate_smoothed` series:**

The `growth_rate_smoothed` series (2-period rolling mean) helps to highlight the underlying trend by reducing the impact of short-term variations.

*   **General Trend:** The smoothed growth rate also shows an initial increase, reaching its highest point around 2010 (`125.76%`), and then a clear deceleration towards 2030 (`43.56%`).
*   **Long-term Patterns:** The smoothing confirms that the rapid acceleration in population growth occurred between the mid-1970s and 2010. After 2010, while the population continued to increase significantly in absolute numbers, the *rate* of growth began to slow down.

**3. Comparison of `growth_rate` with `growth_rate_smoothed`:**

*   The smoothed growth rate follows the general pattern of the original growth rate but presents a less volatile picture. For instance, the minor dip in the original growth rate between 1985 and 1990 is less pronounced in the smoothed series, which shows a continuous increase up to 2010.
*   The smoothed data confirms the peak growth period around 2005-2010 more clearly, and shows a consistent deceleration post-2010, making the long-term trend more apparent.

**4. Significant Shifts and Sustained Periods:**

*   **Sustained Acceleration (1975-2010):** There was a prolonged period of accelerating population growth, with the growth rate consistently increasing, indicating an expanding base and faster percentage increases.
*   **Peak Growth Rate (2005-2010):** This five-year period saw the highest percentage growth in population, indicating the most dynamic phase of demographic change.
*   **Sustained Deceleration (2010-2030):** Following the peak, the growth rate entered a sustained period of deceleration. While the population is still growing substantially in absolute terms, the percentage increase year-over-year is projected to decrease. This could be due to various factors not captured in this data, such as changing birth rates, mortality rates, or migration patterns. However, it indicates a shift from extremely rapid growth to a more moderated, though still strong, growth trajectory.

In summary, Eastern Equatoria State experienced a significant acceleration in its population growth rate from 1975 to 2010, reaching its zenith in the late 2000s. Post-2010, the growth rate has been consistently decelerating, suggesting a transition towards a more stable, albeit still expanding, population dynamic.

In [ ]:
import matplotlib.font_manager as fm

# Clear the matplotlib font cache
fm._load_fontmanager(try_read_cache=False)

In [ ]:
# Plot the population over time with the new font size
plt.figure(figsize=(10, 6))
plt.plot(df['Date'], df['Population'], marker='o')
plt.title('Eastern Equatoria State Population Over Time')
plt.xlabel('Date')
plt.ylabel('Population in Millions')
plt.grid(True, alpha = 0.3, color = 'grey')
plt.show()

In [ ]:
plt.savefig("ssd_population.png", dpi = 365, bbox_inches = 'tight')

### Population Trend Analysis

Based on the `df` DataFrame, the following population trends for Eastern Equatoria State have been observed:

1.  **Initial Population**: In 1975, the estimated population was approximately 11,392.
2.  **Final Population**: By 2030, the projected population is approximately 7,703,000.
3.  **Overall Trend**: There is a clear and dramatic upward trend in population over the entire estimated period, indicating substantial growth.
4.  **Periods of Significant Increase**:
    *   **Early Growth (1975-1990)**: Population grew steadily from roughly 11,000 to 64,000.
    *   **Accelerated Growth (1995-2005)**: The population started to accelerate, crossing 100,000 in 1995 and reaching over 600,000 by 2005.
    *   **Rapid Expansion (2010-2030)**: This period shows the most significant increase, with the population entering the millions. From 2010 (approximately 1.37 million) to 2015 (approximately 2.61 million), the population almost doubled. This rapid growth continues, with projections showing it reaching over 7.7 million by 2030. The growth rate, as previously calculated, indicates significant percentage increases year over year, particularly from the early 2000s onwards.

In summary, Eastern Equatoria State has experienced, and is projected to continue experiencing, very substantial population growth, with the most rapid increases occurring in the 21st century.

# Task
Analyze the overall population trends and growth rates for Eastern Equatoria State over the estimated period, identifying key changes, periods of significant increase, and the dynamics of change, then provide a comprehensive summary of these findings.

## Analyze Population Trends

### Subtask:
Analyze the overall population trends over the estimated period, identifying key changes and periods of significant increase. This will involve reviewing the 'Population' data in the DataFrame.


## Analyze Population Growth Rate

### Subtask:
Examine the calculated population growth rates, both original and smoothed, to understand the dynamics of change. This step will involve interpreting the 'growth_rate' and 'growth_rate_smoothed' series.


## Summarize Key Findings

### Subtask:
Consolidate all observations from the population trend and growth rate analysis into a clear and concise summary, highlighting important insights and conclusions.


## Final Task

### Subtask:
Present the comprehensive summary of the population and growth rate analysis for Eastern Equatoria State.


## Summary:

### Data Analysis Key Findings

*   **Dramatic Population Increase**: Eastern Equatoria State's population is projected to increase dramatically from an estimated 11,392 in 1975 to approximately 7.7 million by 2030, representing an exponential growth pattern.
*   **Periods of Significant Population Growth**:
    *   **Early Growth (1975-1990)**: Population grew steadily from roughly 11,000 to 64,000.
    *   **Accelerated Growth (1995-2005)**: The population surpassed 100,000 by 1995 and exceeded 600,000 by 2005.
    *   **Rapid Expansion (2010-2030)**: This period shows the most significant increase, with the population entering the millions, nearly doubling from approximately 1.37 million in 2010 to about 2.61 million by 2015.
*   **Peak Growth Rate (2000-2010)**: The population growth rate reached its highest during this decade, with the original growth rate peaking at 127.22% in 2005, and the smoothed rate peaking around 125.76% in 2010. This indicates an extremely rapid doubling of the population within five-year intervals during this period.
*   **Projected Deceleration in Growth Rate**: Following its peak, the growth rate began to decelerate. By 2030, the projected growth rate is expected to be around 38-43%, significantly lower than its peak, despite the absolute population continuing to increase substantially.

### Insights or Next Steps

*   The sustained rapid population expansion, despite a decelerating growth *rate*, will require significant planning for resource allocation, infrastructure development, and social services to meet the needs of a rapidly growing population.
*   Further analysis could investigate the underlying factors contributing to the past rapid growth (e.g., birth rates, mortality rates, migration) and the projected deceleration, which would inform more targeted policy interventions.
